Steps:
1. Create Synthetic data for forecast model
2. Use prophet for forcasting 
3. Register the Model in Snowflake Model registry
4. Inferencing using the register model

In [ ]:
# Import python packages
import streamlit as st
import pandas as pd
import numpy as np
from snowflake.snowpark.context import get_active_session
session = get_active_session()
## get prophet model
from prophet import Prophet
from snowflake.ml.registry import Registry
from snowflake.ml.model import custom_model
from snowflake.ml.model import model_signature


In [ ]:
db = 'ML_MODELS'
schema ='DS'
stage_nm = f'{db}.{schema}.MODEL_STAGE'
session.use_warehouse('ML_FS_WH')

In [ ]:
start_date = '2020-01-01'  # Start date
end_date = '2023-01-01'  # End date
freq = 'D'  # Daily frequency
n_periods = pd.to_datetime(end_date) - pd.to_datetime(start_date)
n_periods = n_periods.days  # Total number of days

# Generate a date range
dates = pd.date_range(start=start_date, periods=n_periods, freq=freq)

# Trend component: A simple linear trend
trend = np.linspace(10, 100, n_periods)  # Linear trend (start at 10 and increase to 100)

# Seasonality component: A yearly sinusoidal seasonal effect
seasonal = 10 * np.sin(np.linspace(0, 2 * np.pi, n_periods))

# Noise component: Random noise added for realism
np.random.seed(42)  # To ensure reproducibility
noise = np.random.normal(0, 5, n_periods)  # Gaussian noise with mean=0 and std=5

# Combine all components to form the final target variable
y = trend + seasonal + noise

# Create the DataFrame for Prophet
df = pd.DataFrame({
    'ds': dates,
    'y': y
})



In [ ]:
# write as Snowpark Dataframe and save as table
sdf = session.create_dataframe(df)
sdf.write.mode("overwrite").save_as_table(f"{db}.{schema}.Sales_Data", table_type="transient")

In [ ]:
## future uses, you can direclty use data from snowflake table

my_df = session.table (f"{db}.{schema}.Sales_Data").to_pandas()
my_df.head()

In [ ]:
#Initialize Prophet model
my_forcast_model = Prophet()

my_forcast_model.fit(my_df)

future = my_forcast_model.make_future_dataframe(periods=30)
forecast = my_forcast_model.predict(future)

# View the forecast
print(forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']])


## Serialize the model


In [ ]:
import pickle
import pandas as pd
from snowflake.ml.model import custom_model

pickle.dump(my_forcast_model, open('ProphetModel.pkl', 'wb'))

## Uplad the model into stage
session.file.put("ProphetModel.pkl", f"@{stage_nm}/", auto_compress=False)

In [ ]:
ls @ML_MODELS.DS.MODEL_STAGE

## Logging Model 
The [model registry](https://docs.snowflake.com/en/developer-guide/snowflake-ml/model-registry/bring-your-own-model-types) has a bunch of built-in model types, but you can also log other models, like ones you’ve trained with external tools or grabbed from open-source repositories. As long as they can be serialized and extend the snowflake.ml.model.custom_model.CustomModel class, they’ll work just fine.

Here's how you will log your prophet model

In [ ]:
# Initialize ModelContext with keyword arguments
# my_model can be any supported model type
# my_file_path is a local pickle file path

mc = custom_model.ModelContext(
    artifacts={
        'config': 'ProphetModel.pkl'
    }
)


# Define a custom model class that utilizes the context
class MyProphetModel(custom_model.CustomModel):

    def __init__(self,context:custom_model.ModelContext) -> None:
        super().__init__(context)
        ## use 'file_path to load the piecked object
        with open(self.context['config'],'rb') as f:
            self.model =pickle.load(f)
    @custom_model.inference_api
    def predict(self,X:pd.DataFrame) -> pd.DataFrame:
        X_copy = X.copy()
        X_copy['ds']=pd.to_datetime(X_copy['ds'])# ensure correrct datetime
        forecast = self.model.predict(X_copy)
        res_df = forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]]
        return res_df

Create Test Data

In [ ]:

def create_datedf(max_date, period=7): 
    # Create a new DataFrame starting from max_date + 1 day, adding 7 days to each subsequent row
    date_range = pd.date_range(start=max_date + pd.Timedelta(days=1), periods=period)
    new_df = pd.DataFrame({'ds': date_range})
    return new_df


import pandas as pd
max_date = my_df['ds'].max()
#max_date

df_1 = create_datedf(max_date, period=10)

## prediction on test data
forecast_model = MyProphetModel(mc)

output = forecast_model.predict(df_1)
output.head(5)

## Model registry

In [ ]:
reg = Registry(session,database_name = db,schema_name= schema)

custom_mv = reg.log_model(
   forecast_model,
    model_name="Prophet_forcast_model",
    version_name="v1",
    conda_dependencies=["prophet"],
    sample_input_data= df_1,
    options={'relax_version': False},
    comment = 'My Prophet forcast experiment using the CustomModel API'
)

In [ ]:
reg.show_models()
# Install snowflake-ml-python
from snowflake.ml.registry import registry

reg = registry.Registry(session=session, database_name='ML_MODELS', schema_name='DS')

mv = reg.get_model('PROPHET_FORCAST_MODEL').version('VERSION_1')

pr = mv.run(df_1, function_name='PREDICT')



In [ ]:
pr